# Trump-Related Films — Analysis & NLP

Score analysis and text analysis of OMDb plot descriptions across political perspectives.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from wordcloud import WordCloud

nltk.download('vader_lexicon', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

sns.set_theme(style="whitegrid")
palette = {"critical": "#e74c3c", "pro-trump": "#3498db", "neutral": "#95a5a6"}

## 1. Load and Clean OMDb Data

In [ ]:
df = pd.read_csv("trump-films-omdb.csv")

# Parse scores to numeric
df["rt_score"] = df["RT_Tomatometer"].str.rstrip("%").astype(float)
df["metacritic_score"] = df["Metacritic"].str.split("/").str[0].astype(float)
df["imdb_score"] = pd.to_numeric(df["imdbRating"], errors="coerce")
df["imdb_votes"] = df["imdbVotes"].str.replace(",", "").astype(float)
df["runtime_min"] = df["Runtime"].str.extract(r"(\d+)").astype(float)

# Normalize IMDb to 0-100 scale for comparison
df["imdb_100"] = df["imdb_score"] * 10

# Political perspective of each film
perspective_map = {
    "The Apprentice": "critical",
    "Bad President": "critical",
    "A Storm Foretold": "critical",
    "Fahrenheit 11/9": "critical",
    "Unfit: The Psychology of Donald Trump": "critical",
    "Get Me Roger Stone": "critical",
    "You've Been Trumped": "critical",
    "You've Been Trumped Too": "critical",
    "Trumped: Inside the Greatest Political Upset of All Time": "neutral",
    "Looking for Melania Trump": "critical",
    "A Dangerous Game": "critical",
    "Vindicating Trump": "pro-trump",
    "Trump Card": "pro-trump",
    "The Plot Against the President": "pro-trump",
    "My Dinner with Trump": "pro-trump",
    "Government Gangsters": "pro-trump",
    "The First Step": "neutral",
    "Harry Benson: Shoot First": "neutral",
    "The Comey Rule": "critical",
    "Trump: An American Dream": "neutral",
    "Celebrity": "neutral",
}
df["perspective"] = df["Title"].map(perspective_map)

print(f"{len(df)} films loaded")
print(f"\nPerspective breakdown:")
print(df["perspective"].value_counts().to_string())
print(f"\nScore coverage:")
print(f"  IMDb:       {df['imdb_score'].notna().sum()}/{len(df)}")
print(f"  RT:         {df['rt_score'].notna().sum()}/{len(df)}")
print(f"  Metacritic: {df['metacritic_score'].notna().sum()}/{len(df)}")
df[["Title", "Year", "perspective", "imdb_score", "rt_score", "metacritic_score"]]

## 2. Score Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# IMDb by perspective
sns.stripplot(data=df, x="perspective", y="imdb_score", hue="perspective",
              palette=palette, s=10, ax=axes[0], legend=False)
axes[0].set_title("IMDb Rating by Perspective")
axes[0].set_ylim(2, 10)

# IMDb vs RT (where both exist)
both = df.dropna(subset=["rt_score", "imdb_100"])
for persp, color in palette.items():
    subset = both[both["perspective"] == persp]
    axes[1].scatter(subset["imdb_100"], subset["rt_score"], c=color, s=80, label=persp, zorder=3)
axes[1].plot([20, 100], [20, 100], "--", color="gray", alpha=0.5, zorder=1)
axes[1].set_xlabel("IMDb (scaled 0-100)")
axes[1].set_ylabel("RT Tomatometer")
axes[1].set_title("IMDb vs Rotten Tomatoes")
axes[1].legend()

# Vote count by perspective (log scale)
sns.boxplot(data=df, x="perspective", y="imdb_votes", palette=palette, ax=axes[2])
axes[2].set_yscale("log")
axes[2].set_title("IMDb Vote Count by Perspective")

plt.tight_layout()
plt.savefig("score-analysis.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Summary stats by perspective
summary = df.groupby("perspective").agg(
    count=("Title", "count"),
    imdb_mean=("imdb_score", "mean"),
    imdb_median=("imdb_score", "median"),
    rt_mean=("rt_score", "mean"),
    metacritic_mean=("metacritic_score", "mean"),
    median_votes=("imdb_votes", "median"),
).round(1)

print("Score summary by perspective:")
summary

## 3. Critic Coverage Gap

Which films get reviewed by professional critics at all?

In [ ]:
df["has_rt"] = df["rt_score"].notna()
df["has_metacritic"] = df["metacritic_score"].notna()

coverage = df.groupby("perspective").agg(
    total=("Title", "count"),
    with_rt=("has_rt", "sum"),
    with_metacritic=("has_metacritic", "sum"),
)
coverage["rt_pct"] = (coverage["with_rt"] / coverage["total"] * 100).round(0)
coverage["mc_pct"] = (coverage["with_metacritic"] / coverage["total"] * 100).round(0)

print("Critic coverage by perspective:")
print(coverage.to_string())
print()
print("Pro-Trump films with NO critic score:")
print(df[(df["perspective"] == "pro-trump") & (~df["has_rt"])]["Title"].to_string(index=False))

## 4. NLP on Plot Descriptions

OMDb provides a plot summary for each film. These are written by third parties (IMDb contributors), not the filmmakers, so they reflect how each film is *described* — which itself varies by political framing.

In [ ]:
sia = SentimentIntensityAnalyzer()

df["plot_sentiment"] = df["Plot"].apply(
    lambda x: sia.polarity_scores(str(x))["compound"]
)
df["plot_pos"] = df["Plot"].apply(
    lambda x: sia.polarity_scores(str(x))["pos"]
)
df["plot_neg"] = df["Plot"].apply(
    lambda x: sia.polarity_scores(str(x))["neg"]
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Sentiment by perspective
sns.stripplot(data=df, x="perspective", y="plot_sentiment", hue="perspective",
              palette=palette, s=10, ax=axes[0], legend=False)
axes[0].axhline(0, color="gray", linestyle="--", alpha=0.5)
axes[0].set_title("Plot Description Sentiment by Perspective")
axes[0].set_ylabel("VADER Compound Score")

# Positive vs negative language
for persp, color in palette.items():
    subset = df[df["perspective"] == persp]
    axes[1].scatter(subset["plot_pos"], subset["plot_neg"], c=color,
                    s=80, label=persp, zorder=3)
axes[1].plot([0, 0.4], [0, 0.4], "--", color="gray", alpha=0.5)
axes[1].set_xlabel("Positive Language (proportion)")
axes[1].set_ylabel("Negative Language (proportion)")
axes[1].set_title("Positive vs Negative Language in Plots")
axes[1].legend()

plt.tight_layout()
plt.savefig("plot-sentiment.png", dpi=150, bbox_inches="tight")
plt.show()

print("Mean plot sentiment by perspective:")
print(df.groupby("perspective")["plot_sentiment"].agg(["mean", "median", "count"]).round(3))

In [ ]:
# Show the actual plot text with sentiment for context
for _, row in df.sort_values("plot_sentiment").iterrows():
    print(f"[{row['plot_sentiment']:+.3f}] ({row['perspective']}) {row['Title']}")
    print(f"  {row['Plot'][:120]}..." if len(str(row['Plot'])) > 120 else f"  {row['Plot']}")
    print()

## 5. Word Clouds by Perspective

In [ ]:
stop_words = set(stopwords.words("english"))
stop_words.update(["trump", "donald", "president", "film", "movie",
                   "documentary", "one", "also", "would"])

def tokenize_clean(text):
    tokens = word_tokenize(str(text).lower())
    return [t for t in tokens if t.isalpha() and t not in stop_words and len(t) > 2]

df["tokens"] = df["Plot"].apply(tokenize_clean)

perspectives = ["critical", "pro-trump", "neutral"]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, persp in zip(axes, perspectives):
    subset = df[df["perspective"] == persp]
    all_tokens = [t for tokens in subset["tokens"] for t in tokens]
    
    if len(all_tokens) < 3:
        ax.set_title(f"{persp} (insufficient text)")
        ax.axis("off")
        continue
    
    wc = WordCloud(width=600, height=400, background_color="white",
                   colormap="RdBu_r", max_words=40)
    wc.generate(" ".join(all_tokens))
    ax.imshow(wc, interpolation="bilinear")
    ax.set_title(f"{persp} ({len(subset)} films)")
    ax.axis("off")

plt.suptitle("Most Frequent Words in Plot Descriptions by Perspective", fontsize=14)
plt.tight_layout()
plt.savefig("wordclouds.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. TF-IDF: Distinguishing Language by Perspective

What words and bigrams are most distinctive to how critical vs. pro-Trump vs. neutral films describe their subject?

In [ ]:
# Combine all plot text per perspective
perspective_texts = df.groupby("perspective")["Plot"].apply(
    lambda x: " ".join(x.dropna())
).to_dict()

labels = list(perspective_texts.keys())
texts = [perspective_texts[l] for l in labels]

tfidf = TfidfVectorizer(max_features=200, stop_words="english",
                         ngram_range=(1, 2), min_df=1)
tfidf_matrix = tfidf.fit_transform(texts)
feature_names = tfidf.get_feature_names_out()

print("Top 15 most distinctive terms per perspective:\n")
for i, label in enumerate(labels):
    scores = tfidf_matrix[i].toarray().flatten()
    top_idx = scores.argsort()[-15:][::-1]
    terms = [(feature_names[j], round(scores[j], 3)) for j in top_idx]
    print(f"--- {label.upper()} ---")
    for term, score in terms:
        print(f"  {term:30s} {score}")
    print()

## 7. Score vs. Sentiment Correlation

Do films with more negative plot descriptions get higher or lower ratings?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Plot sentiment vs IMDb
for persp, color in palette.items():
    subset = df[df["perspective"] == persp]
    axes[0].scatter(subset["plot_sentiment"], subset["imdb_score"],
                    c=color, s=80, label=persp, zorder=3)
axes[0].set_xlabel("Plot Sentiment (VADER)")
axes[0].set_ylabel("IMDb Rating")
axes[0].set_title("Plot Sentiment vs IMDb Rating")
axes[0].legend()

# Plot sentiment vs RT (where available)
has_rt = df.dropna(subset=["rt_score"])
for persp, color in palette.items():
    subset = has_rt[has_rt["perspective"] == persp]
    axes[1].scatter(subset["plot_sentiment"], subset["rt_score"],
                    c=color, s=80, label=persp, zorder=3)
axes[1].set_xlabel("Plot Sentiment (VADER)")
axes[1].set_ylabel("RT Tomatometer")
axes[1].set_title("Plot Sentiment vs RT Score")
axes[1].legend()

plt.tight_layout()
plt.savefig("sentiment-vs-score.png", dpi=150, bbox_inches="tight")
plt.show()

# Correlations
print("Correlations with plot sentiment:")
print(f"  IMDb:       r={df['plot_sentiment'].corr(df['imdb_score']):.3f} (n={df['imdb_score'].notna().sum()})")
print(f"  RT:         r={df['plot_sentiment'].corr(df['rt_score']):.3f} (n={df['rt_score'].notna().sum()})")
print(f"  Metacritic: r={df['plot_sentiment'].corr(df['metacritic_score']):.3f} (n={df['metacritic_score'].notna().sum()})")

## 8. Full Dataset Export

In [ ]:
export_cols = ["Title", "Year", "Type", "Genre", "Director", "runtime_min",
               "perspective", "imdb_score", "imdb_votes", "rt_score",
               "metacritic_score", "BoxOffice", "plot_sentiment",
               "plot_pos", "plot_neg", "Plot"]
df[export_cols].to_csv("trump-films-analyzed.csv", index=False)
print(f"Exported {len(df)} films to trump-films-analyzed.csv")
df[export_cols].head(21)